# 14. Custom Transformations, Mapping & Apply: Beginner Guide

### 🌟 What Are Custom Transformations, Mapping & Apply in Pandas?
When standard arithmetic operations aren't enough, Pandas provides tools to apply custom logic: **`.map()`** for dictionary/value replacements, **`.replace()`** for targeted substitutions, and **`.apply()`** for custom row/column transformations.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Row/Column Mapping**: Covers `.apply(func, axis=0/1)`.
- **Series Dictionary/Function Mapping**: Covers `.map(dict_or_func)`.
- **Element-Wise Application**: Covers `df.map()` element-wise transformation.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### 🔹 Row-Wise Logic with `.apply()`
Applies a custom multi-feature risk scoring rule across transaction rows. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** `.apply()` runs a standard Python loop row-by-row. Whenever possible, use built-in vectorized operations (`df['a'] + df['b']`) which run up to 100x faster!.

**Syntax:** `df.apply(lambda row: ..., axis=1)`


In [2]:
def score_risk(row):
    return 'High_Risk' if row['transaction_amount'] > 500 and row['account_age_months'] < 6 else 'Normal'

df_sample = df.head(100).copy()
df_sample['risk_category'] = df_sample.apply(score_risk, axis=1)
print('Row-Wise Risk Scoring Head:\n', df_sample[['transaction_id', 'transaction_amount', 'account_age_months', 'risk_category']].head(3))

Row-Wise Risk Scoring Head:
   transaction_id  transaction_amount  account_age_months risk_category
0       TX110686             1216.33                  56        Normal
1       TX107170              324.99                 112        Normal
2       TX108328              136.66                  68        Normal


### 🔹 Series Dictionary Mapping with `.map()`
Maps card types to standard processing fee percentages. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** `.apply()` runs a standard Python loop row-by-row. Whenever possible, use built-in vectorized operations (`df['a'] + df['b']`) which run up to 100x faster!.

**Syntax:** `df['card_type'].map({'Visa': 0.015, 'MasterCard': 0.018, 'Amex': 0.025, 'Discover': 0.012})`


In [3]:
fee_map = {'Visa': 0.015, 'MasterCard': 0.018, 'Amex': 0.025, 'Discover': 0.012}
df_sample['interchange_fee'] = df_sample['card_type'].map(fee_map)
print('Mapped Interchange Fees Head:\n', df_sample[['card_type', 'interchange_fee']].head(3))

Mapped Interchange Fees Head:
     card_type  interchange_fee
0        Visa            0.015
1  MasterCard            0.018
2    Discover            0.012


### 🔹 Element-Wise Transformations with `df.map()`
Applies formatting to all numeric columns across a DataFrame slice. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** `.apply()` runs a standard Python loop row-by-row. Whenever possible, use built-in vectorized operations (`df['a'] + df['b']`) which run up to 100x faster!.

**Syntax:** `df[['col1', 'col2']].map(lambda x: f'{x:.2f}')`


In [4]:
num_slice = df[['transaction_amount', 'account_age_months']].head(3)
formatted_slice = num_slice.map(lambda val: f'{val:.1f}')
print('Element-Wise Mapped Table:\n', formatted_slice)

Element-Wise Mapped Table:
   transaction_amount account_age_months
0             1216.3               56.0
1              325.0              112.0
2              136.7               68.0


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: High-Speed Vectorization vs Slow `.apply(axis=1)`

**Approach:** Benchmark `.apply(axis=1)` vs `np.where()` on 15,000 transaction rows to prove 50x-100x vectorization speedup.
**Syntax:** `np.where((df['amt'] > 500) & (df['age'] < 6), 'High', 'Normal')`


In [5]:
t0 = time.perf_counter()
res_apply = df.head(1000).apply(score_risk, axis=1)
t_apply = time.perf_counter() - t0

t0 = time.perf_counter()
res_vec = np.where((df['transaction_amount'].head(1000) > 500) & (df['account_age_months'].head(1000) < 6), 'High_Risk', 'Normal')
t_vec = time.perf_counter() - t0
print(f'apply(axis=1): {t_apply*1000:.2f} ms')
print(f'Vectorized np.where: {t_vec*1000:.2f} ms ({t_apply/t_vec:.1f}x speedup)')

apply(axis=1): 5.29 ms
Vectorized np.where: 0.46 ms (11.6x speedup)
